In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType


#Define expected schema

schema = StructType([
    StructField("store_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("snapshot_ts", StringType(), True)
    
])

df = (
    spark.readStream
    .format("cloudfiles")
    .option("cloudfiles.format", "json")
    .option("cloudfiles.schemaLocation", "/Volumes/retailnew/bronze/schema_logs/inventory")
    .schema(schema)
    .load("/Volumes/retailnew/bronze/raw_data/inventory")
    .withColumn("ingest_ts", F.current_timestamp())
       
)
(
    df.writeStream
    .format("Delta")
    .option("checkpointLocation", "/Volumes/retailnew/bronze/checkpoints/inventory")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table("retailnew.bronze.inventory")
)